In [1]:
import os
import re
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# --- Configuration ---
X_vals = ["jobs1"]
Y_checkpoints = [
    "checkpoint_0.04.pth", "checkpoint_0.08.pth", "checkpoint_0.12.pth",
    "checkpoint_0.17.pth", "checkpoint_0.21.pth", "checkpoint_0.25.pth",
    "checkpoint_0.29.pth", "checkpoint_0.33.pth", "checkpoint_0.37.pth",
    "checkpoint_0.42.pth", "checkpoint_0.46.pth", "checkpoint_0.50.pth",
    "checkpoint_0.54.pth", "checkpoint_0.58.pth", "checkpoint_0.62.pth",
    "checkpoint_0.67.pth", "checkpoint_0.71.pth", "checkpoint_0.75.pth",
    "checkpoint_0.79.pth", "checkpoint_0.83.pth", "checkpoint_0.87.pth",
    "checkpoint_0.92.pth", "checkpoint_0.96.pth", "checkpoint_1.00.pth"
]

Z_tasks = ["ID", "semantic_preserving_ID"]

A_config = {
    "0.1": Path("./results"),
    "1": Path("./results"),
    "10": Path("./results"),
    "20": Path("./results_extended"),
    "40": Path("./results_extended"),
    "80": Path("./results_extended")   # path stays "80", only display label changes
}
A_vals_ordered = ["0.1", "1", "10", "20", "40", "80"]

# Display label override: key = internal A value, value = what to print on the plot
A_display_labels = {
    "0.1": "0.1",
    "1":   "1",
    "10":  "10",
    "20":  "20",
    "40":  "40",
    "80":  "70",   # <-- show 70% on the plot, keep path as "80"
}

# --- Font Size Palette ---
FONT_TITLE       = 17
FONT_AXIS_LABELS = 16
FONT_TICK_LABELS = 16
FONT_LEGEND      = 13

# ── Changed output folder ──
output_dir = Path("plots_pfe")
output_dir.mkdir(exist_ok=True)

def get_accuracy_percentage(base_dir, x, y, z, a):
    folder_name = f"eval_{x}_{y}_{z}_p_{a}"
    file_path = base_dir / folder_name / "model-eval.log"
    if not file_path.exists():
        return 0.0
    try:
        with open(file_path, 'r') as f:
            content = f.read()
            match = re.search(r"hard-accuracy:\s*\d+\s*=\s*([\d.]+)%", content)
            if match:
                return float(match.group(1))
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
    return 0.0

# 1. Collect all data
print("Collecting data...")
rows = []
for a, base_dir in A_config.items():
    for x in X_vals:
        for i, y in enumerate(Y_checkpoints):
            step_val = i + 1
            data_point = {'A': a, 'Job': x, 'Step': step_val}
            for z in Z_tasks:
                data_point[z] = get_accuracy_percentage(base_dir, x, y, z, a)
            rows.append(data_point)

df = pd.DataFrame(rows)

# ──────────────────────────────────────────────
# 2a. Main 2×3 grid plot (all 6 percentages)
# ──────────────────────────────────────────────
print("Generating grid chart...")
for x in X_vals:
    fig, axes = plt.subplots(3,2, figsize=(18,19))
    axes = axes.flatten()

    for idx, a in enumerate(A_vals_ordered):
        ax = axes[idx]
        subset = df[(df['A'] == a) & (df['Job'] == x)].sort_values('Step').copy()
        subset['Semantic_Adjusted'] = (subset['semantic_preserving_ID'] * subset['ID']) / 100.0

        ax.plot(subset['Step'], subset['ID'],
                color='black', linestyle='-', linewidth=1.5,
                label='Accuracy on ID data')
        ax.plot(subset['Step'], subset['Semantic_Adjusted'],
                color='blue', linestyle='-', marker='o', markersize=5, linewidth=1.5,
                label='Accuracy on replaced variable names')

        # Use display label (80 → 70)
        display_label = A_display_labels[a]
        ax.set_title(f"Variable diversity percentage : {display_label}%",
                     fontsize=FONT_TITLE, pad=10)

        if idx >= 4:
            ax.set_xlabel("Training progress by data size (Millions)", fontsize=FONT_AXIS_LABELS)
        ax.set_xlim(0, 24)
        ax.set_xticks(range(0, 25, 4))
        ax.tick_params(axis='x', labelsize=FONT_TICK_LABELS)

        if idx % 2 == 0:
            ax.set_ylabel("Accuracy (%)", fontsize=FONT_AXIS_LABELS)
        ax.set_ylim(-5, 105)
        ax.set_yticks(range(0, 101, 10))
        ax.tick_params(axis='y', labelsize=FONT_TICK_LABELS)

        ax.grid(True, linestyle='--', color='lightgray', alpha=0.7)

        if a in ["0.1", "1"]:
            ax.legend(loc='center right', fontsize=FONT_LEGEND)
        else:
            ax.legend(loc='lower right', fontsize=FONT_LEGEND)

    plt.tight_layout()
    filename = f"PFE_EXAMPLE{x}.png"
    plt.savefig(output_dir / filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved grid → {output_dir / filename}")

# ──────────────────────────────────────────────
# 2b. Standalone plot for 1% (no title)
# ──────────────────────────────────────────────
print("Generating standalone 1% chart...")
for x in X_vals:
    fig, ax = plt.subplots(figsize=(9, 6))

    subset = df[(df['A'] == "1") & (df['Job'] == x)].sort_values('Step').copy()
    subset['Semantic_Adjusted'] = (subset['semantic_preserving_ID'] * subset['ID']) / 100.0

    ax.plot(subset['Step'], subset['ID'],
            color='black', linestyle='-', linewidth=1.5,
            label='Accuracy on ID data')
    ax.plot(subset['Step'], subset['Semantic_Adjusted'],
            color='blue', linestyle='-', marker='o', markersize=5, linewidth=1.5,
            label='Accuracy on replaced variable names')

    # No title
    ax.set_xlabel("Training progress by data size (Millions)", fontsize=FONT_AXIS_LABELS)
    ax.set_ylabel("Accuracy (%)", fontsize=FONT_AXIS_LABELS)
    ax.set_xlim(0, 24)
    ax.set_xticks(range(0, 25, 4))
    ax.tick_params(axis='x', labelsize=FONT_TICK_LABELS)
    ax.set_ylim(-5, 105)
    ax.set_yticks(range(0, 101, 10))
    ax.tick_params(axis='y', labelsize=FONT_TICK_LABELS)
    ax.grid(True, linestyle='--', color='lightgray', alpha=0.7)
    ax.legend(loc='center right', fontsize=FONT_LEGEND)

    plt.tight_layout()
    filename = f"plot_standalone_1pct_{x}.png"
    plt.savefig(output_dir / filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved standalone → {output_dir / filename}")

print(f"\nDone! All plots saved in '{output_dir}' at 300 DPI.")

Generating grid chart...


Saved grid → plots_pfe/PFE_EXAMPLEjobs1.png
Generating standalone 1% chart...
Saved standalone → plots_pfe/plot_standalone_1pct_jobs1.png

Done! All plots saved in 'plots_pfe' at 300 DPI.


In [1]:
import os
import re
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# --- Configuration ---
X_vals = ["jobs1"]
Y_checkpoints = [
    "checkpoint_0.04.pth", "checkpoint_0.08.pth", "checkpoint_0.12.pth",
    "checkpoint_0.17.pth", "checkpoint_0.21.pth", "checkpoint_0.25.pth",
    "checkpoint_0.29.pth", "checkpoint_0.33.pth", "checkpoint_0.37.pth",
    "checkpoint_0.42.pth", "checkpoint_0.46.pth", "checkpoint_0.50.pth",
    "checkpoint_0.54.pth", "checkpoint_0.58.pth", "checkpoint_0.62.pth",
    "checkpoint_0.67.pth", "checkpoint_0.71.pth", "checkpoint_0.75.pth",
    "checkpoint_0.79.pth", "checkpoint_0.83.pth", "checkpoint_0.87.pth",
    "checkpoint_0.92.pth", "checkpoint_0.96.pth", "checkpoint_1.00.pth"
]

Z_tasks = ["ID", "semantic_preserving_ID"]

A_config = {
    "0.1": Path("./results"),
    "1": Path("./results"),
    "10": Path("./results"),
    "20": Path("./results_extended"),
    "40": Path("./results_extended"),
    "80": Path("./results_extended")   # path stays "80", only display label changes
}
A_vals_ordered = ["0.1", "1", "10", "20", "40", "80"]

# Display label override: key = internal A value, value = what to print on the plot
A_display_labels = {
    "0.1": "0.1",
    "1":   "1",
    "10":  "10",
    "20":  "20",
    "40":  "40",
    "80":  "70",   # <-- show 70% on the plot, keep path as "80"
}

# --- Font Size Palette ---
FONT_TITLE       = 17
FONT_AXIS_LABELS = 16
FONT_TICK_LABELS = 16
FONT_LEGEND      = 13

# ── Changed output folder ──
output_dir = Path("plots_pfe")
output_dir.mkdir(exist_ok=True)

def get_accuracy_percentage(base_dir, x, y, z, a):
    folder_name = f"eval_{x}_{y}_{z}_p_{a}"
    file_path = base_dir / folder_name / "model-eval.log"
    if not file_path.exists():
        return 0.0
    try:
        with open(file_path, 'r') as f:
            content = f.read()
            match = re.search(r"hard-accuracy:\s*\d+\s*=\s*([\d.]+)%", content)
            if match:
                return float(match.group(1))
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
    return 0.0

# 1. Collect all data
print("Collecting data...")
rows = []
for a, base_dir in A_config.items():
    for x in X_vals:
        for i, y in enumerate(Y_checkpoints):
            step_val = i + 1
            data_point = {'A': a, 'Job': x, 'Step': step_val}
            for z in Z_tasks:
                data_point[z] = get_accuracy_percentage(base_dir, x, y, z, a)
            rows.append(data_point)

df = pd.DataFrame(rows)

# ──────────────────────────────────────────────
# 2a. Main 2×3 grid plot (all 6 percentages)
# ──────────────────────────────────────────────
print("Generating grid chart...")
for x in X_vals:
    fig, axes = plt.subplots(2, 3, figsize=(20, 9))
    axes = axes.flatten()

    for idx, a in enumerate(A_vals_ordered):
        ax = axes[idx]
        subset = df[(df['A'] == a) & (df['Job'] == x)].sort_values('Step').copy()
        subset['Semantic_Adjusted'] = (subset['semantic_preserving_ID'] * subset['ID']) / 100.0

        ax.plot(subset['Step'], subset['ID'],
                color='black', linestyle='-', linewidth=1.5,
                label='Accuracy on ID data')
        ax.plot(subset['Step'], subset['Semantic_Adjusted'],
                color='blue', linestyle='-', marker='o', markersize=5, linewidth=1.5,
                label='Accuracy on replaced variable names')

        # Use display label (80 → 70)
        display_label = A_display_labels[a]
        ax.set_title(f"Variable diversity percentage : {display_label}%",
                     fontsize=FONT_TITLE, pad=10)

        if idx >= 3:
            ax.set_xlabel("Training progress by data size (Millions)", fontsize=FONT_AXIS_LABELS)
        ax.set_xlim(0, 24)
        ax.set_xticks(range(0, 25, 4))
        ax.tick_params(axis='x', labelsize=FONT_TICK_LABELS)

        if idx % 3 == 0:
            ax.set_ylabel("Accuracy (%)", fontsize=FONT_AXIS_LABELS)
        ax.set_ylim(-5, 105)
        ax.set_yticks(range(0, 101, 10))
        ax.tick_params(axis='y', labelsize=FONT_TICK_LABELS)

        ax.grid(True, linestyle='--', color='lightgray', alpha=0.7)

        if a in ["0.1", "1"]:
            ax.legend(loc='center right', fontsize=FONT_LEGEND)
        else:
            ax.legend(loc='lower right', fontsize=FONT_LEGEND)

    plt.tight_layout()
    filename = f"plot_grid_{x}.png"
    plt.savefig(output_dir / filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved grid → {output_dir / filename}")

# ──────────────────────────────────────────────
# 2b. Standalone plot for 1% (no title)
# ──────────────────────────────────────────────
print("Generating standalone 1% chart...")
for x in X_vals:
    fig, ax = plt.subplots(figsize=(9, 6))

    subset = df[(df['A'] == "1") & (df['Job'] == x)].sort_values('Step').copy()
    subset['Semantic_Adjusted'] = (subset['semantic_preserving_ID'] * subset['ID']) / 100.0

    ax.plot(subset['Step'], subset['ID'],
            color='black', linestyle='-', linewidth=1.5,
            label='Accuracy on ID data')
    ax.plot(subset['Step'], subset['Semantic_Adjusted'],
            color='blue', linestyle='-', marker='o', markersize=5, linewidth=1.5,
            label='Accuracy on replaced variable names')

    # No title
    ax.set_xlabel("Training progress by data size (Millions)", fontsize=FONT_AXIS_LABELS)
    ax.set_ylabel("Accuracy (%)", fontsize=FONT_AXIS_LABELS)
    ax.set_xlim(0, 24)
    ax.set_xticks(range(0, 25, 4))
    ax.tick_params(axis='x', labelsize=FONT_TICK_LABELS)
    ax.set_ylim(-5, 105)
    ax.set_yticks(range(0, 101, 10))
    ax.tick_params(axis='y', labelsize=FONT_TICK_LABELS)
    ax.grid(True, linestyle='--', color='lightgray', alpha=0.7)
    ax.legend(loc='center right', fontsize=FONT_LEGEND)

    plt.tight_layout()
    filename = f"plot_standalone_1pct_{x}.png"
    plt.savefig(output_dir / filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved standalone → {output_dir / filename}")

print(f"\nDone! All plots saved in '{output_dir}' at 300 DPI.")

Generating grid chart...
Saved grid → plots_pfe/plot_grid_jobs1.png
Generating standalone 1% chart...
Saved standalone → plots_pfe/plot_standalone_1pct_jobs1.png

Done! All plots saved in 'plots_pfe' at 300 DPI.


In [ ]:
import os
import re
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# --- Configuration ---
X_vals = ["jobs1"]
Y_checkpoints = [
    "checkpoint_0.04.pth", "checkpoint_0.08.pth", "checkpoint_0.12.pth",
    "checkpoint_0.17.pth", "checkpoint_0.21.pth", "checkpoint_0.25.pth",
    "checkpoint_0.29.pth", "checkpoint_0.33.pth", "checkpoint_0.37.pth",
    "checkpoint_0.42.pth", "checkpoint_0.46.pth", "checkpoint_0.50.pth",
    "checkpoint_0.54.pth", "checkpoint_0.58.pth", "checkpoint_0.62.pth",
    "checkpoint_0.67.pth", "checkpoint_0.71.pth", "checkpoint_0.75.pth",
    "checkpoint_0.79.pth", "checkpoint_0.83.pth", "checkpoint_0.87.pth",
    "checkpoint_0.92.pth", "checkpoint_0.96.pth", "checkpoint_1.00.pth"
]

# We only focus on the two requested tasks
Z_tasks = ["ID", "semantic_preserving_ID"]

# Map each percentage 'A' to its corresponding directory
A_config = {
    "0.1": Path("./results"),
    "1": Path("./results"),
    "10": Path("./results"),
    "20": Path("./results_extended"),
    "40": Path("./results_extended"),
    "80": Path("./results_extended")
}
A_vals_ordered = ["0.1", "1", "10", "20", "40", "80"]

# --- Font Size Palette ---
# Modify these values to scale the text anywhere in the charts
FONT_TITLE = 17
FONT_AXIS_LABELS = 16
FONT_TICK_LABELS = 16
FONT_LEGEND = 13

output_dir = Path("plot_final")
output_dir.mkdir(exist_ok=True)

def get_accuracy_percentage(base_dir, x, y, z, a):
    """Parses hard-accuracy from the log file and returns it as a percentage (0-100)."""
    folder_name = f"eval_{x}_{y}_{z}_p_{a}"
    file_path = base_dir / folder_name / "model-eval.log"
    
    if not file_path.exists():
        return 0.0
    
    try:
        with open(file_path, 'r') as f:
            content = f.read()
            # Find "hard-accuracy: ... = X.XX%"
            match = re.search(r"hard-accuracy:\s*\d+\s*=\s*([\d.]+)%", content)
            if match:
                return float(match.group(1))
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
    return 0.0

# 1. Collect all data into a DataFrame
print("Collecting data...")
rows = []
for a, base_dir in A_config.items():
    for x in X_vals:
        for i, y in enumerate(Y_checkpoints):
            # Map the 24 checkpoints to Steps 1 through 24 for the X-axis
            step_val = i + 1 
            
            data_point = {'A': a, 'Job': x, 'Step': step_val}
            for z in Z_tasks:
                data_point[z] = get_accuracy_percentage(base_dir, x, y, z, a)
            rows.append(data_point)

df = pd.DataFrame(rows)

# 2. Plotting
print("Generating charts...")
for x in X_vals:
    # Changed figsize to (20, 7) for a wider, shorter aspect ratio
    # that scales perfectly to PDF document widths without taking up the whole page.
    fig, axes = plt.subplots(2, 3, figsize=(20, 9))
    
    # Flatten axes array for easy iteration
    axes = axes.flatten()
    
    for idx, a in enumerate(A_vals_ordered):
        ax = axes[idx]
        
        # Filter data for this specific percentage and job
        subset = df[(df['A'] == a) & (df['Job'] == x)].sort_values('Step').copy()
        
        # Calculate Adjusted Semantic Accuracy
        subset['Semantic_Adjusted'] = (subset['semantic_preserving_ID'] * subset['ID']) / 100.0
        
        # Plot ID (Matching image style: solid black line)
        ax.plot(subset['Step'], subset['ID'], color='black', linestyle='-', linewidth=1.5, 
                label='Accuracy on ID data')
        
        # Plot Semantic Adjusted (Matching image style: solid blue line with circular markers)
        ax.plot(subset['Step'], subset['Semantic_Adjusted'], color='blue', linestyle='-', 
                marker='o', markersize=5, linewidth=1.5, label='Accuracy on replaced variable names')
        
        # Format Title 
        ax.set_title(f"Variable diversity percentage : {a}%", fontsize=FONT_TITLE, pad=10)
        
        # Format X-axis
        if idx >= 3:  # Only add X label to the bottom row to keep it clean
            ax.set_xlabel("Training progress by data size (Millions)", fontsize=FONT_AXIS_LABELS)
        ax.set_xlim(0, 24)
        ax.set_xticks(range(0, 25, 4)) # Ticks at 0, 4, 8, 12, 16, 20, 24
        ax.tick_params(axis='x', labelsize=FONT_TICK_LABELS)
        
        # Format Y-axis
        if idx % 3 == 0:  # Only add Y label to the left-most columns
            ax.set_ylabel("Accuracy (%)", fontsize=FONT_AXIS_LABELS)
        ax.set_ylim(-5, 105) # Slight padding to match image bounds visually
        ax.set_yticks(range(0, 101, 10)) # Ticks at 0, 10, 20... 100
        ax.tick_params(axis='y', labelsize=FONT_TICK_LABELS)
        
        # Clean Grid matching the image
        ax.grid(True, linestyle='--', color='lightgray', alpha=0.7)
        
        # Legend (Conditional placement)
        if a in ["0.1", "1"]:
            ax.legend(loc='center right', fontsize=FONT_LEGEND)
        else:
            ax.legend(loc='lower right', fontsize=FONT_LEGEND)
        
    plt.tight_layout()
    
    # Save the composite plot with HIGH QUALITY (dpi=300)
    filename = f"plot_grid_{x}.png"
    plt.savefig(output_dir / filename, dpi=300, bbox_inches='tight')
    plt.close()

print(f"Done! Subplot grid saved in the '{output_dir}' folder at 300 DPI.")

Generating charts...


Done! Subplot grid saved in the 'plot_final' folder at 300 DPI.


In [3]:
import os
import re
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# --- Configuration ---
X_vals = ["jobs1"]
Y_checkpoints = [
    "checkpoint_0.04.pth", "checkpoint_0.08.pth", "checkpoint_0.12.pth",
    "checkpoint_0.17.pth", "checkpoint_0.21.pth", "checkpoint_0.25.pth",
    "checkpoint_0.29.pth", "checkpoint_0.33.pth", "checkpoint_0.37.pth",
    "checkpoint_0.42.pth", "checkpoint_0.46.pth", "checkpoint_0.50.pth",
    "checkpoint_0.54.pth", "checkpoint_0.58.pth", "checkpoint_0.62.pth",
    "checkpoint_0.67.pth", "checkpoint_0.71.pth", "checkpoint_0.75.pth",
    "checkpoint_0.79.pth", "checkpoint_0.83.pth", "checkpoint_0.87.pth",
    "checkpoint_0.92.pth", "checkpoint_0.96.pth", "checkpoint_1.00.pth"
]

# We only focus on the two requested tasks
Z_tasks = ["ID", "semantic_preserving_ID"]

# Map each percentage 'A' to its corresponding directory
A_config = {
    "0.1": Path("./results"),
    "1": Path("./results"),
    "10": Path("./results"),
    "20": Path("./results_extended"),
    "40": Path("./results_extended"),
    "80": Path("./results_extended")
}
A_vals_ordered = ["0.1", "1", "10", "20", "40", "80"]

output_dir = Path("plot_final")
output_dir.mkdir(exist_ok=True)

def get_accuracy_percentage(base_dir, x, y, z, a):
    """Parses hard-accuracy from the log file and returns it as a percentage (0-100)."""
    folder_name = f"eval_{x}_{y}_{z}_p_{a}"
    file_path = base_dir / folder_name / "model-eval.log"
    
    if not file_path.exists():
        return 0.0
    
    try:
        with open(file_path, 'r') as f:
            content = f.read()
            # Find "hard-accuracy: ... = X.XX%"
            match = re.search(r"hard-accuracy:\s*\d+\s*=\s*([\d.]+)%", content)
            if match:
                # We return the raw percentage (e.g., 95.5) instead of dividing by 100
                return float(match.group(1))
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
    return 0.0

# 1. Collect all data into a DataFrame
print("Collecting data...")
rows = []
for a, base_dir in A_config.items():
    for x in X_vals:
        for i, y in enumerate(Y_checkpoints):
            # Map the 24 checkpoints to Steps 1 through 24 for the X-axis
            step_val = i + 1 
            
            data_point = {'A': a, 'Job': x, 'Step': step_val}
            for z in Z_tasks:
                data_point[z] = get_accuracy_percentage(base_dir, x, y, z, a)
            rows.append(data_point)

df = pd.DataFrame(rows)

# 2. Plotting
print("Generating charts...")
for x in X_vals:
    # Create a giant rectangle: 2 rows, 3 columns
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    # Optional overall title if needed: fig.suptitle(f"Accuracy Over Training Progress - {x}", fontsize=16)
    
    # Flatten axes array for easy iteration
    axes = axes.flatten()
    
    for idx, a in enumerate(A_vals_ordered):
        ax = axes[idx]
        
        # Filter data for this specific percentage and job
        subset = df[(df['A'] == a) & (df['Job'] == x)].sort_values('Step')
        
        # Plot ID
        ax.plot(subset['Step'], subset['ID'], 'k-', label='Accuracy on ID data', linewidth=2)
        
        # Plot Semantic Preserving ID
        ax.plot(subset['Step'], subset['semantic_preserving_ID'], 'r--', 
                 label='Accuracy on unseen variable names', linewidth=2)
        
        # Format Title
        ax.set_title(f"variable diversity percentage : {a}%", pad=10)
        
        # Format X-axis
        if idx >= 3:  # Only add X label to the bottom row to keep it clean
            ax.set_xlabel("Training progress by data size (millions)")
        ax.set_xlim(0, 24)
        ax.set_xticks(range(0, 25, 4)) # Ticks at 0, 4, 8, 12, 16, 20, 24
        
        # Format Y-axis
        if idx % 3 == 0:  # Only add Y label to the left-most columns
            ax.set_ylabel("Accuracy (%)")
        ax.set_ylim(0, 100)
        ax.set_yticks(range(0, 101, 10)) # Ticks at 0, 10, 20... 100
        
        ax.legend(loc='lower right')
        ax.grid(True, linestyle='--', alpha=0.6)
        
    plt.tight_layout()
    
    # Save the composite plot
    filename = f"plot_grid_{x}.png"
    plt.savefig(output_dir / filename)
    plt.close()

print(f"Done! Subplot grids saved in the '{output_dir}' folder.")

Generating charts...
Done! Subplot grids saved in the 'plot_final' folder.


In [2]:
import os
import re
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# --- Increase Global Font Sizes ---
plt.rcParams.update({
    'font.size': 18,          # Global fallback font size
    'axes.titlesize': 24,     # Chart title font size
    'axes.labelsize': 20,     # X and Y axis label font size
    'xtick.labelsize': 16,    # X-axis tick numbers font size
    'ytick.labelsize': 16,    # Y-axis tick numbers font size
    'legend.fontsize': 16     # Legend font size
})

# --- Configuration ---
X_vals = ["jobs1"]
Y_checkpoints = [
    "checkpoint_0.04.pth", "checkpoint_0.08.pth", "checkpoint_0.12.pth",
    "checkpoint_0.17.pth", "checkpoint_0.21.pth", "checkpoint_0.25.pth",
    "checkpoint_0.29.pth", "checkpoint_0.33.pth", "checkpoint_0.37.pth",
    "checkpoint_0.42.pth", "checkpoint_0.46.pth", "checkpoint_0.50.pth",
    "checkpoint_0.54.pth", "checkpoint_0.58.pth", "checkpoint_0.62.pth",
    "checkpoint_0.67.pth", "checkpoint_0.71.pth", "checkpoint_0.75.pth",
    "checkpoint_0.79.pth", "checkpoint_0.83.pth", "checkpoint_0.87.pth",
    "checkpoint_0.92.pth", "checkpoint_0.96.pth", "checkpoint_1.00.pth"
]

# Trimmed down to only what is needed for the requested charts
Z_tasks = ["ID", "semantic_preserving_ID"]

# 'a' represents your percentage value
A_vals = ["80", "40", "20", "10", "1", "0.1"]

# Path setup
results_dir = Path("./results")
output_dir = Path("plotsss")
output_dir.mkdir(exist_ok=True)

def get_accuracy(x, y, z, a):
    """Parses hard-accuracy from the log file as a percentage (0-100)."""
    folder_name = f"eval_{x}_{y}_{z}_p_{a}"
    file_path = results_dir / folder_name / "model-eval.log"
    
    if not file_path.exists():
        print(f"Missing file: {file_path}")
        return 0.0
    
    try:
        with open(file_path, 'r') as f:
            content = f.read()
            # Find "hard-accuracy: ... = 0.00%"
            match = re.search(r"hard-accuracy:\s*\d+\s*=\s*([\d.]+)%", content)
            if match:
                # MODIFIED: Removed the / 100.0 to keep it as a 0-100 percentage
                return float(match.group(1)) 
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
    return 0.0

# 1. Collect all data into a DataFrame
print("Collecting data...")
rows = []
for a in A_vals:
    for x in X_vals:
        for y in Y_checkpoints:
            # Use the actual checkpoint index (1 to 24)
            step_val = Y_checkpoints.index(y) + 1
            
            data_point = {'A': a, 'Job': x, 'Step': step_val}
            for z in Z_tasks:
                data_point[z] = get_accuracy(x, y, z, a)
            rows.append(data_point)

df = pd.DataFrame(rows)

# 2. Plotting
print("Generating charts...")
for a in A_vals:
    for x in X_vals:
        # Filter data for this specific combination and sort by 'Step' integer
        subset = df[(df['A'] == a) & (df['Job'] == x)].sort_values('Step')
        
        plt.figure(figsize=(10, 6))
        
        # Plot ID Baseline using 'Step' for the X-axis
        plt.plot(subset['Step'], subset['ID'], 'k-', label='ID Accuracy', linewidth=2)
        
        # Plot Hidden Variables using 'Step' for the X-axis
        plt.plot(subset['Step'], subset['semantic_preserving_ID'], 'b-o', label='OOD_Hidden_variables Accuracy')
        
        plt.title(f"Variable diversity percentage : {a}%")
        
        # --- MODIFIED LABELS AND LIMITS ---
        plt.xlabel("Training progress by data size (Millions)")
        plt.ylabel("Accuracy (%)")
        
        # Y-axis scaling from 0 to 100 (with slight padding)
        plt.ylim(-5, 105)
        # Force Y-axis ticks to show 0, 10, 20, ..., 100
        plt.yticks(range(0, 101, 10))
        
        # Force X-axis ticks to print every 4 checkpoints (0, 4, 8, 12, 16, 20, 24)
        plt.xticks(range(0, 25, 4))
        
        plt.legend(loc='center right')
        plt.grid(True, linestyle='--', alpha=0.6)
        
        # Save the plot
        filename = f"plot_A{a}_{x}_OOD_Hidden_variables.png"
        plt.savefig(output_dir / filename)
        plt.close()

print(f"Done! Plots saved in the '{output_dir}' folder.")

Missing file: results/eval_jobs1_checkpoint_0.04.pth_ID_p_80/model-eval.log
Missing file: results/eval_jobs1_checkpoint_0.04.pth_semantic_preserving_ID_p_80/model-eval.log
Missing file: results/eval_jobs1_checkpoint_0.08.pth_ID_p_80/model-eval.log
Missing file: results/eval_jobs1_checkpoint_0.08.pth_semantic_preserving_ID_p_80/model-eval.log
Missing file: results/eval_jobs1_checkpoint_0.12.pth_ID_p_80/model-eval.log
Missing file: results/eval_jobs1_checkpoint_0.12.pth_semantic_preserving_ID_p_80/model-eval.log
Missing file: results/eval_jobs1_checkpoint_0.17.pth_ID_p_80/model-eval.log
Missing file: results/eval_jobs1_checkpoint_0.17.pth_semantic_preserving_ID_p_80/model-eval.log
Missing file: results/eval_jobs1_checkpoint_0.21.pth_ID_p_80/model-eval.log
Missing file: results/eval_jobs1_checkpoint_0.21.pth_semantic_preserving_ID_p_80/model-eval.log
Missing file: results/eval_jobs1_checkpoint_0.25.pth_ID_p_80/model-eval.log
Missing file: results/eval_jobs1_checkpoint_0.25.pth_semantic_pr

In [ ]:
import os
import re
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# --- Configuration ---
X_vals = ["jobs1", "jobs2"]
Y_checkpoints = [
    "checkpoint_0.04.pth", "checkpoint_0.08.pth", "checkpoint_0.12.pth",
    "checkpoint_0.17.pth", "checkpoint_0.21.pth", "checkpoint_0.25.pth",
    "checkpoint_0.29.pth", "checkpoint_0.33.pth", "checkpoint_0.37.pth",
    "checkpoint_0.42.pth", "checkpoint_0.46.pth", "checkpoint_0.50.pth",
    "checkpoint_0.54.pth", "checkpoint_0.58.pth", "checkpoint_0.62.pth",
    "checkpoint_0.67.pth", "checkpoint_0.71.pth", "checkpoint_0.75.pth",
    "checkpoint_0.79.pth", "checkpoint_0.83.pth", "checkpoint_0.87.pth",
    "checkpoint_0.92.pth", "checkpoint_0.96.pth", "checkpoint_1.00.pth"
]
Z_tasks = [
    "ID", "OOD_Deeper", "OOD_Denser", "OOD_Hidden_numbers",
    "OOD_Hidden_variables", "OOD_Longer_snippets",
    "OOD_Longer_variables", "OOD_Mixed_ops", "semantic_preserving_ID"
]
A_vals = ["10", "1", "0.1", "0.01"]

# Path setup: script is in simple_alpha, data is in ../results
results_dir = Path("./results")
output_dir = Path("plotsss")
output_dir.mkdir(exist_ok=True)

def get_accuracy(x, y, z, a):
    """Parses hard-accuracy from the log file."""
    folder_name = f"eval_{x}_{y}_{z}_p_{a}"
    file_path = results_dir / folder_name / "model-eval.log"
    
    if not file_path.exists():
        print("waaaaaaaa")
        return 0.0
    
    try:
        with open(file_path, 'r') as f:
            content = f.read()
            # Find "hard-accuracy: ... = 0.00%"
            match = re.search(r"hard-accuracy:\s*\d+\s*=\s*([\d.]+)%", content)
            if match:
                return float(match.group(1)) / 100.0
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
    return 0.0

# 1. Collect all data into a DataFrame
print("Collecting data...")
rows = []
for a in A_vals:
    for x in X_vals:
        for y in Y_checkpoints:
            # Fix: extract only the float part (0.04) from "checkpoint_0.04.pth"
            checkpoint_match = re.search(r"checkpoint_([\d.]+)\.pth", y)
            checkpoint_val = float(checkpoint_match.group(1)) if checkpoint_match else 0.0
            
            data_point = {'A': a, 'Job': x, 'Checkpoint': checkpoint_val}
            for z in Z_tasks:
                data_point[z] = get_accuracy(x, y, z, a)
            rows.append(data_point)

df = pd.DataFrame(rows)

# 2. Plotting
print("Generating charts...")
for a in A_vals:
    for x in X_vals:
        # Filter data for this specific combination
        subset = df[(df['A'] == a) & (df['Job'] == x)].sort_values('Checkpoint')
        
        # Calculate Adjusted Semantic Accuracy: Semantic * ID
        subset['Semantic_Adjusted'] = subset['semantic_preserving_ID'] * subset['ID']
        
        # Iterate through Z tasks (excluding semantic_preserving_ID as the main focus)
        # because we plot it alongside others or alongside ID specifically.
        for z in Z_tasks[:-1]:  # Excludes 'semantic_preserving_ID'
            plt.figure(figsize=(10, 6))
            
            # Plot ID Baseline (always present)
            plt.plot(subset['Checkpoint'], subset['ID'], 'k-', label='ID Accuracy', linewidth=2)
            
            # Plot Semantic Adjusted (always present)
            plt.plot(subset['Checkpoint'], subset['Semantic_Adjusted'], 'r--', 
                     label='Semantic ID (Adjusted: ID * Semantic)', alpha=0.8)
            
            # If the current task is NOT ID, plot it as well
            if z != "ID":
                plt.plot(subset['Checkpoint'], subset[z], 'b-o', label=f'{z} Accuracy')
                plt.title(f"Task: {z} vs ID | Job: {x}, p: {a}")
            else:
                plt.title(f"ID and Semantic Accuracy | Job: {x}, p: {a}")

            plt.xlabel("Training Progress (Checkpoint)")
            plt.ylabel("Accuracy (0.0 - 1.0)")
            plt.ylim(-0.05, 1.05)
            plt.legend(loc='lower right')
            plt.grid(True, linestyle='--', alpha=0.6)
            
            # Save the plot
            filename = f"plot_A{a}_{x}_{z}.png"
            plt.savefig(output_dir / filename)
            plt.close()

print(f"Done! Plots saved in the '{output_dir}' folder.")

Generating charts...


Done! Plots saved in the 'plots' folder.


In [1]:

import os
import re
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

# --- Configuration ---
X_vals = ["jobs1"]
Y_checkpoints = [
    "checkpoint_0.04.pth", "checkpoint_0.08.pth", "checkpoint_0.12.pth",
    "checkpoint_0.17.pth", "checkpoint_0.21.pth", "checkpoint_0.25.pth",
    "checkpoint_0.29.pth", "checkpoint_0.33.pth", "checkpoint_0.37.pth",
    "checkpoint_0.42.pth", "checkpoint_0.46.pth", "checkpoint_0.50.pth",
    "checkpoint_0.54.pth", "checkpoint_0.58.pth", "checkpoint_0.62.pth",
    "checkpoint_0.67.pth", "checkpoint_0.71.pth", "checkpoint_0.75.pth",
    "checkpoint_0.79.pth", "checkpoint_0.83.pth", "checkpoint_0.87.pth",
    "checkpoint_0.92.pth", "checkpoint_0.96.pth", "checkpoint_1.00.pth"
]

Z_tasks = ["ID","OOD_Hidden_variables", "semantic_preserving_ID"]
# Z_tasks = [
#     "ID", "OOD_Deeper", "OOD_Denser", "OOD_Hidden_numbers",
#     "OOD_Hidden_variables", "OOD_Longer_snippets",
#     "OOD_Longer_variables", "OOD_Mixed_ops", "semantic_preserving_ID"
# ]
A_vals = ["80"]

# Path setup: script is in simple_alpha, data is in ../results
results_dir = Path("./results_extended")
output_dir = Path("plots")
output_dir.mkdir(exist_ok=True)

def get_accuracy(x, y, z, a):
    """Parses hard-accuracy from the log file."""
    folder_name = f"eval_{x}_{y}_{z}_p_{a}"
    file_path = results_dir / folder_name / "model-eval.log"
    
    if not file_path.exists():
        # print("waaaaaaaa") # Commented out to keep output clean, but logic remains
        return 0.0
    
    try:
        with open(file_path, 'r') as f:
            content = f.read()
            # Find "hard-accuracy: ... = 0.00%"
            match = re.search(r"hard-accuracy:\s*\d+\s*=\s*([\d.]+)%", content)
            if match:
                return float(match.group(1)) / 100.0
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
    return 0.0

# 1. Collect all data into a DataFrame
print("Collecting data...")
rows = []
for a in A_vals:
    for x in X_vals:
        for y in Y_checkpoints:
            # Fix: extract only the float part (0.04) from "checkpoint_0.04.pth"
            checkpoint_match = re.search(r"checkpoint_([\d.]+)\.pth", y)
            checkpoint_val = float(checkpoint_match.group(1)) if checkpoint_match else 0.0
            
            data_point = {'A': a, 'Job': x, 'Checkpoint': checkpoint_val}
            for z in Z_tasks:
                data_point[z] = get_accuracy(x, y, z, a)
            rows.append(data_point)

df = pd.DataFrame(rows)

# 2. Plotting
print("Generating charts...")
for a in A_vals:
    for x in X_vals:
        # Filter data for this specific combination
        subset = df[(df['A'] == a) & (df['Job'] == x)].sort_values('Checkpoint')
        
        # Calculate Adjusted Semantic Accuracy: Semantic * ID
        subset['Semantic_Adjusted'] = subset['semantic_preserving_ID'] * subset['ID']

        # --- NEW FOLDER STRUCTURE LOGIC ---
        # Create directory: plots/job/percentage (e.g., plots/jobs1/10/)
        current_save_dir = output_dir / x / a
        current_save_dir.mkdir(parents=True, exist_ok=True)
        # ----------------------------------
        
        # Iterate through Z tasks (excluding semantic_preserving_ID as the main focus)
        for z in Z_tasks[:-1]:  # Excludes 'semantic_preserving_ID'
            plt.figure(figsize=(10, 6))
            
            # Plot ID Baseline (always present)
            plt.plot(subset['Checkpoint'], subset['ID'], 'k-', label='ID Accuracy', linewidth=2)
            
            # Plot Semantic Adjusted (always present)
            plt.plot(subset['Checkpoint'], subset['Semantic_Adjusted'], 'r--', 
                     label='Semantic ID (Adjusted: ID * Semantic)', alpha=0.8)
            
            # If the current task is NOT ID, plot it as well
            if z != "ID":
                plt.plot(subset['Checkpoint'], subset[z], 'b-o', label=f'{z} Accuracy')
                plt.title(f"Task: {z} vs ID | Job: {x}, p: {a}")
            else:
                plt.title(f"ID and Semantic Accuracy | Job: {x}, p: {a}")

            plt.xlabel("Training Progress (Checkpoint)")
            plt.ylabel("Accuracy (0.0 - 1.0)")
            plt.ylim(-0.05, 1.05)
            plt.legend(loc='lower right')
            plt.grid(True, linestyle='--', alpha=0.6)
            
            # Save the plot into the new hierarchical folder
            # Filename is just "ID.png", "OOD_Deeper.png", etc.
            plt.savefig(current_save_dir / f"{z}.png")
            plt.close()

print(f"Done! Plots saved in the '{output_dir}' folder structure.")

Generating charts...
Done! Plots saved in the 'plots' folder structure.
